**TP 5 - Nettoyage d'un dataset de ventes**

*Louison Mahe*

In [259]:
#Importation des librairies
import numpy as np
import pandas as pd

**Exercice 1 - Première exploration**

In [260]:
#importation du csv
ventes = pd.read_csv('ventes_brut .csv', sep=',', header=0, index_col=0)

In [261]:
#Combien de lignes et de colonnes le dataset contient-il ?
#Il y a 228 lignes et 8 colonnes
ventes.shape

(228, 8)

In [262]:
#Quels sont les types de chaque colonne ? Sont-ils cohérents avec ce qu'ils devraient être ?
#ils sont tous cohérents sauf le prix unitaire qui devrait être un float et non un string
ventes.dtypes

date_vente           str
produit              str
categorie            str
prix_unitaire        str
quantite         float64
client               str
region               str
mode_paiement        str
dtype: object

In [263]:
#Que remarquez-vous en affichant les 10 premières lignes ?
#La date et le prix unitaire ne sont pas normalisés
#Certaines valeurs sont manquantes
#Parfois, l'ecriture est en majuscule, parfois en minuscule 
#il y a des valeurs incoherente : quantite negative
ventes.head(10)

,date_vente,produit,categorie,prix_unitaire,quantite,client,region,mode_paiement
id_vente,,,,,,,,
1,2025-10-07,Casque audio,électronique,NaN,-1.0,Client_1152,nouvelle aquitaine,Carte bancaire
2,2025-09-11,Casque audio,Electronique,"185,41 €",4.0,Client_1099,Île-de-France,Carte bancaire
3,28/11/2025,Stylo pack x10,fournitures,7.55,1.0,Client_1115,Auvergne-Rhône-Alpes,Espèces
4,2025-12-02,Chaise de bureau,mobilier,292.49,-1.0,Client_1128,IDF,Espèces
5,2025-12-10,Lampe LED,Mobilier,57.04,1.0,Client_1114,AURA,Chèque
6,2025-05-27,Bureau réglable,MOBILIER,"311,61 €",3.0,Client_1129,IDF,Chèque
7,2025-05-12,Stylo pack x10,Fourniture,7.72,3.0,Client_1165,Occitanie,Virement
8,2025-11-05,Ordinateur portable,électronique,"1187,64 €",4.0,Client_1004,Auvergne-Rhône-Alpes,Chèque
9,2025-05-11,Ordinateur portable,ELECTRONIQUE,1214.54,-1.0,Client_1101,NaN,Espèces


**Exercice 2 - Diagnostic qualité**

In [264]:
#Nombre de valeurs manquantes par colonne
ventes.isna().sum()

date_vente        0
produit           0
categorie        15
prix_unitaire    13
quantite         15
client            0
region           13
mode_paiement    13
dtype: int64

In [265]:
#Nombre de lignes strictement dupliquées
print(ventes.duplicated().sum())

8


In [266]:
#Valeurs uniques des colonnes categorie et region — que constatez-vous ?
#Il y a des doublons ecrit de maniere differente
print(ventes['categorie'].unique())
print(ventes['region'].unique())

<StringArray>
[ 'électronique', 'Electronique ',  'fournitures ',      'mobilier',
      'Mobilier',      'MOBILIER',    'Fourniture',  'ELECTRONIQUE',
   'Fournitures',  'Électronique',             nan]
Length: 11, dtype: str
<StringArray>
[  'nouvelle aquitaine',        'Île-de-France', 'Auvergne-Rhône-Alpes',
                  'IDF',                 'AURA',            'Occitanie',
                    nan,             'bretagne',           'occitanie ',
        'ile de france',             'Bretagne',   'Nouvelle-Aquitaine']
Length: 12, dtype: str


In [267]:
#Valeurs uniques de quantite — voyez-vous des valeurs suspectes ?
#Les valeurs -1 et 0 sont suspectes dans le cadre d'un logiciel de vente
#La quantité vendu doit forcement être superieur à 0
ventes['quantite'].unique()

array([-1.,  4.,  1.,  3.,  2.,  0., nan])

**Exercice 3 - Traiter doublons et valeurs manquantes**

In [268]:
#Supprimez les lignes strictement dupliquées
ventes.drop_duplicates(inplace=True)
ventes.duplicated().sum()

np.int64(0)

In [269]:
#Traitez les valeurs manquantes de prix_unitaire et quantite
#Dans le cadre d'une vente, si il n'y a pas de prix ou de quantité, la vente n'a pas lieu
#Je vais juste supprimer les lignes qui ont des valeurs manquantes pour ces deux colonnes
ventes.dropna(subset=['prix_unitaire', 'quantite'], inplace=True)

In [270]:
#Traitez les valeurs manquantes de categorie, region et mode_paiement
#Ces colonnes sont des informations qui ne sont pas essentielles
#Je vais remplacer les valeurs manquantes par "inconnu"
ventes['mode_paiement'] = ventes['mode_paiement'].fillna("inconnu")
ventes['categorie'] = ventes['categorie'].fillna("inconnu")
ventes['region'] = ventes['region'].fillna("inconnu")

**Exercice 4 - Corriger les types de données**

In [271]:
#prix_unitaire contient parfois du texte (ex : "20,50 €") — transformez toute la colonne en nombres décimaux
ventes['prix_unitaire'] = (
    ventes['prix_unitaire']
    .str.replace('€', '',regex=False)
    .str.replace(',', '.',regex=False)
    .str.strip()
    .astype(float)
)

In [272]:
#date_vente mélange deux formats de date — uniformisez la colonne en type date
ventes['date_vente'] = pd.to_datetime(ventes['date_vente'], format="mixed",dayfirst=True)


In [273]:
#quantite contient des valeurs impossibles (0 ou négatives) — traitez-les, puis convertissez la colonne en entier
#On supprime les lignes qui ont une quantité inférieur ou égale à 0
ventes = ventes[ventes['quantite'] > 0]
ventes['quantite'] = ventes['quantite'].astype(int)

**Exercice 5 - Normaliser les catégories**

In [274]:
#Les colonnes categorie et region contiennent plusieurs orthographes différentes pour une même valeur (majuscules, espaces, abréviations).
#Normalisez-les pour qu'une même réalité soit toujours écrite de la même façon.
correspondance = {
    'idf': 'ile de france',
    'aura': 'auvergne rhone alpes'
}


ventes['categorie'] = (
    ventes['categorie']
    .str.strip()
    .str.lower()
    .str.replace('é', 'e', regex=False)
)

ventes['region'] = (
    ventes['region']
    .str.strip()
    .str.lower()
    .str.replace('é', 'e', regex=False)
    .str.replace('ô', 'o', regex=False)
    .str.replace('î', 'i', regex=False)
    .str.replace('-', ' ', regex=False)
    .replace(correspondance)  #J'utilse replace plutot que .map car map donne Null si la valeur n'est pas dans le dictionnaire
)
ventes['region'].unique()

<StringArray>
[       'ile de france', 'auvergne rhone alpes',            'occitanie',
             'bretagne',   'nouvelle aquitaine',              'inconnu']
Length: 6, dtype: str

**Exercice 6 - Finalisation**

In [275]:
#Créez une colonne montant_total = prix_unitaire × quantite
ventes['montant_total'] = ventes['prix_unitaire'] * ventes['quantite']

In [276]:
#Sauvegardez le résultat dans ventes_propre.csv
ventes.to_csv('ventes_propre.csv')